# 03 - WoE and IV for Win/Loss

This notebook runs variable-wise optimal binning for `Opportunity Result` (Won/Loss), computes WoE bins, and reports Information Value (IV) for all candidate predictors.

In [1]:
import pandas as pd
import numpy as np
from optbinning import OptimalBinning

pd.set_option('display.max_columns', 200)

In [2]:
df = pd.read_excel('../cars.xlsx')
df['target_win'] = df['Opportunity Result'].map({'Won': 1, 'Loss': 0})
print('shape:', df.shape)
print('target nulls:', df['target_win'].isna().sum())

shape: (78025, 20)
target nulls: 0


## Fit WoE/IV by variable

In [3]:
excluded = {'Opportunity Result', 'target_win', 'Opportunity Number'}
features = [c for c in df.columns if c not in excluded]

results = []
failed = []
for col in features:
    x = df[col]
    y = df['target_win']
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(x) else 'categorical'

    try:
        optb = OptimalBinning(name=col, dtype=dtype)
        optb.fit(x, y)
        bt = optb.binning_table.build()
        iv_total = float(bt.iloc[-1]['IV']) if 'IV' in bt.columns else np.nan
        js_total = float(bt.iloc[-1]['JS']) if 'JS' in bt.columns else np.nan
        results.append({
            'variable': col,
            'dtype': dtype,
            'status': optb.status,
            'iv': iv_total,
            'js': js_total
        })
    except Exception as e:
        failed.append({'variable': col, 'error': str(e)[:200]})

iv_df = pd.DataFrame(results)
if not iv_df.empty:
    iv_df = iv_df.sort_values('iv', ascending=False).reset_index(drop=True)
failed_df = pd.DataFrame(failed)
print('processed:', len(iv_df), 'failed:', len(failed_df))
iv_df.head(20)

processed: 17 failed: 0


,variable,dtype,status,iv,js
0,Total Days Identified Through Qualified,numerical,OPTIMAL,0.801425,0.094950
1,Total Days Identified Through Closing,numerical,OPTIMAL,0.692787,0.082768
2,Revenue From Client Past Two Years (USD),categorical,OPTIMAL,0.570736,0.064250
3,Ratio Days Identified To Total Days,numerical,OPTIMAL,0.397489,0.044744
4,Opportunity Amount USD,numerical,OPTIMAL,0.332396,0.039679
5,Ratio Days Qualified To Total Days,numerical,OPTIMAL,0.315554,0.038723
6,Deal Size Category (USD),categorical,OPTIMAL,0.274840,0.033554
7,Ratio Days Validated To Total Days,numerical,OPTIMAL,0.229913,0.028174
8,Sales Stage Change Count,numerical,OPTIMAL,0.119074,0.014761
9,Supplies Subgroup,categorical,OPTIMAL,0.065706,0.008162


## IV interpretation bands

In [4]:
def iv_band(iv):
    if iv < 0.02:
        return 'Not useful'
    if iv < 0.1:
        return 'Weak'
    if iv < 0.3:
        return 'Medium'
    if iv < 0.5:
        return 'Strong'
    return 'Suspicious/Very strong'

iv_df['iv_band'] = iv_df['iv'].apply(iv_band)
iv_df

,variable,dtype,status,iv,js,iv_band
0,Total Days Identified Through Qualified,numerical,OPTIMAL,0.801425,0.094950,Suspicious/Very strong
1,Total Days Identified Through Closing,numerical,OPTIMAL,0.692787,0.082768,Suspicious/Very strong
2,Revenue From Client Past Two Years (USD),categorical,OPTIMAL,0.570736,0.064250,Suspicious/Very strong
3,Ratio Days Identified To Total Days,numerical,OPTIMAL,0.397489,0.044744,Strong
4,Opportunity Amount USD,numerical,OPTIMAL,0.332396,0.039679,Strong
5,Ratio Days Qualified To Total Days,numerical,OPTIMAL,0.315554,0.038723,Strong
6,Deal Size Category (USD),categorical,OPTIMAL,0.274840,0.033554,Medium
7,Ratio Days Validated To Total Days,numerical,OPTIMAL,0.229913,0.028174,Medium
8,Sales Stage Change Count,numerical,OPTIMAL,0.119074,0.014761,Medium
9,Supplies Subgroup,categorical,OPTIMAL,0.065706,0.008162,Weak


In [5]:
iv_df.to_csv('../data/processed/win_loss_iv_report.csv', index=False)
if not failed_df.empty:
    failed_df.to_csv('../data/processed/win_loss_iv_failed.csv', index=False)
print('saved: ../data/processed/win_loss_iv_report.csv')
print('saved failures:', (not failed_df.empty))

saved: ../data/processed/win_loss_iv_report.csv
saved failures: False


## Top variable WoE table example

In [6]:
if iv_df.empty:
    print('No successful WoE/IV fits to display.')
else:
    top_var = iv_df.loc[0, 'variable']
    x = df[top_var]
    y = df['target_win']
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(x) else 'categorical'
    optb = OptimalBinning(name=top_var, dtype=dtype)
    optb.fit(x, y)
    woe_table = optb.binning_table.build()
    print('Top variable:', top_var)
    display(woe_table)

Top variable: Total Days Identified Through Qualified


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.50)",9559,0.122512,5193,4366,0.456742,-1.05806,0.171097,0.020442
1,"[0.50, 3.50)",9394,0.120397,5193,4201,0.447200,-1.019535,0.155324,0.018616
2,"[3.50, 5.50)",5403,0.069247,3509,1894,0.350546,-0.614884,0.030345,0.003734
3,"[5.50, 7.50)",4854,0.062211,3495,1359,0.279975,-0.28694,0.005518,0.000687
4,"[7.50, 9.50)",4786,0.061339,3655,1131,0.236314,-0.05853,0.000213,0.000027
5,"[9.50, 12.50)",6041,0.077424,4845,1196,0.197980,0.16744,0.002071,0.000259
6,"[12.50, 15.50)",5650,0.072413,4889,761,0.134690,0.628586,0.023744,0.002920
7,"[15.50, 18.50)",5003,0.064120,4383,620,0.123926,0.724245,0.027083,0.003313
8,"[18.50, 23.50)",7347,0.094162,6675,672,0.091466,1.064342,0.077052,0.009201
9,"[23.50, 28.50)",5606,0.071849,5115,491,0.087585,1.111964,0.063197,0.007516
